# Reconocimiento de Entidades Nombradas con BERT

Se va a utilizar un transformer BERT para la tarea de reconocimiento de entidades (NER). 

NER se puede ver como una tarea de clasificación de tokens, de forma que el objetivo es asignar una etiqueta a cada token. ¿Cuáles son las etiquetas? Las etiquetas pueden ser las del estándar IOB, donde

    B-type representa un token que es el primer token de una entidad de tipo type.
    I-type representa un token que pertenece a una entidad, pero no es el primero de sus tokens.
    O se utiliza para representar tokens que no pertenecen a ninguna entidad.


In [1]:
!pip install -q datasets transformers[torch]


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


## Cargar el dataset

En este ejercicio utilizaremos uno de los conjuntos de datos más populares para la tarea de reconocimiento de entidades nombradas, el conjunto CoLL-2003 (https://huggingface.co/datasets/conll2003). Este dataset es una colección de noticias de Reuters anotadas con entidades como Persona, Ubicación, Organización y Varios.

In [2]:
from datasets import load_dataset
from datasets import load_dataset_builder

ds_name = "conll2003"

ds = load_dataset_builder(ds_name)
ds

dataset_dict = load_dataset(ds_name)
dataset_dict

/home/madrueno/test-transformers/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

El dataset está distribuido con tres splits: train, validation y test. Además, cada instancia contiene las siguientes características:

    id: el identificador del texto
    tokens: la lista de tokens del texto.
    pos_tags: la lista de etiquetas PoS (categoría gramátical) de los tokens del texto.
    chunk_tags: la lista de etiquetas en formato IOB para representar sintagmas nominales.
    ner_tags: la lista de etiquetas NER, en formatio IOB, para los tokens en el texto.


In [3]:
ner_tags = dataset_dict["train"].features["ner_tags"]
LABELS = ner_tags.feature.names
LABELS

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [4]:
# Cada label está codificada como un entero
idx2label={}
label2idx={}
for index, label in enumerate(LABELS):
    print(label)
    label2idx.update([(label, index)])
    idx2label.update([(index, label)])

print(label2idx)

O
B-PER
I-PER
B-ORG
I-ORG
B-LOC
I-LOC
B-MISC
I-MISC
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}


In [5]:
# Si se ejecuta este código varias veces se verán distintas oraciones con su etiquetado NER
idx2label={}
label2idx={}
for index, label in enumerate(LABELS):
    print(label)
    label2idx.update([(label, index)])
    idx2label.update([(index, label)])

print(label2idx)

O
B-PER
I-PER
B-ORG
I-ORG
B-LOC
I-LOC
B-MISC
I-MISC
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}


## Tokenización

Hay que transformar y utilizar el mismo formato uque necesita BERT: (input_ids, input_type_ids y attention_mask).
La versión cased de BERT es mejor para tareas NER, mientras que uncased es mejor para tareas de clasificación de textos.


In [6]:
from transformers import AutoTokenizer

model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

Ejemplo de la tokenización con BERT. Antes de aplicar el tokenizador a toda la colección. En la siguiente celda, codificamos los tokens del primer texto. El método tokens() nos permite obtener la lista de tokens. Podemos ver que se han agregado dos tokens especiales, [CLS] y [SEP]. Además, las palabras que no están en el vocabulario se han dividido en subpalabras. Por ejemplo, 'lamb' se dividió en 'la' y '##mb'.


El método word_ids() nos permite conocer el índice de cada token en el texto original. Los tokens especiales están indexados con None. El token 'EU' tiene el id 0, mientras que los tokens 'la' y '##mb" corresponden a la palabra en la posición 7.

In [7]:
# Función para asignar correctamente a cada token/subtoken su etiqueta NER correspondiente. 
# En el ejemplo anterior, tanto 'la' como '#mb" deben anotarse con O.
def align_labels_with_tokens(word_ids, tags):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)     # tokens especials se codifican con -100
        elif word_id != current_word:   # pasamos a un nuevo token
            current_word = word_id
            new_labels.append(tags[word_id])
        else:                           # estamos en el mismo token
            label = tags[current_word]
            # ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
            #  0        1       2       3       4           5       6       7           8
            if label % 2 != 0:       # si la label es impar, quiere. decir que el anterior era el primero,
                                     # debemos pasar a I- (es la siguiente)
                new_labels.append(label+1)
            else:
                new_labels.append(label)


    return new_labels

In [8]:
# Se comprueba para las cinco primeras oraciones del split de training
for i in range(5):
    print("\nSentence: ", str(i))
    tags = dataset_dict["train"][i]["ner_tags"]
    inputs = tokenizer(dataset_dict["train"][i]["tokens"], is_split_into_words=True)
    word_ids = inputs.word_ids()

    aligned_labels = align_labels_with_tokens(word_ids, tags)
    print(inputs.tokens())
    print(aligned_labels)
    for t in aligned_labels:
        if t != -100:
            print(idx2label[t], end = ' ')
        else:
            print(str(-100), end = ' ')

    print()



Sentence:  0
['[CLS]', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'la', '##mb', '.', '[SEP]']
[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]
-100 B-ORG O B-MISC O O O B-MISC O O O -100 

Sentence:  1
['[CLS]', 'Peter', 'Blackburn', '[SEP]']
[-100, 1, 2, -100]
-100 B-PER I-PER -100 

Sentence:  2
['[CLS]', 'BR', '##US', '##SE', '##LS', '1996', '-', '08', '-', '22', '[SEP]']
[-100, 5, 6, 6, 6, 0, 0, 0, 0, 0, -100]
-100 B-LOC I-LOC I-LOC I-LOC O O O O O -100 

Sentence:  3
['[CLS]', 'The', 'European', 'Commission', 'said', 'on', 'Thursday', 'it', 'disagreed', 'with', 'German', 'advice', 'to', 'consumers', 'to', 's', '##hun', 'British', 'la', '##mb', 'until', 'scientists', 'determine', 'whether', 'mad', 'cow', 'disease', 'can', 'be', 'transmitted', 'to', 'sheep', '.', '[SEP]']
[-100, 0, 3, 4, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]
-100 O B-ORG I-ORG O O O O O O B-MISC O O O O O O B-MISC O O O O O O O O O O O O O O O -10

In [ ]:
# función recibe como entrada un conjunto de datos, aplica el tokenizador y alinea las etiquetas del dataset con esta tokenización. 
# Esta función añade un nuevo campo al conjunto de datos que contiene las etiquetas alineadas (lo vamos a llamar labels).

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        #get the ids for the instance i
        word_ids = tokenized_inputs.word_ids(i)
        #align the words with their corresponding labels
        new_labels.append(align_labels_with_tokens(word_ids, labels))

    # we add a new feature to the dataset with the aligned labels for each instance
    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

tokenized_datasets = dataset_dict.map(tokenize_and_align_labels, batched=True, remove_columns=dataset_dict["train"].column_names)

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Map: 100%|██████████| 3453/3453 [00:00<00:00, 14807.59 examples/s]


## Data Collator

DataCollatorForTokenClassification que directamente añade padding a todos los campos de la entrada: input_ids, token_type_ids, attention_mask y labels.



In [10]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

2025-05-10 00:47:45.559788: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746838065.580778  117665 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746838065.587284  117665 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746838065.604452  117665 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746838065.604467  117665 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746838065.604470  117665 computation_placer.cc:177] computation placer alr

In [11]:
# Se aplica el colector a las primeras 5 instancias del conjunto de datos de entrenamiento
batch = data_collator([tokenized_datasets["train"][i] for i in range(5)])
batch["labels"]


tensor([[-100,    3,    0,    7,    0,    0,    0,    7,    0,    0,    0, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100],
        [-100,    1,    2, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100],
        [-100,    5,    6,    6,    6,    0,    0,    0,    0,    0, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100],
        [-100,    0,    3,    4,    0,    0,    0,    0,    0,    0,    7,    0,
            0,    0,    0,    0,    0,    7,    0,    0,    0,    0,    0,    0,
            0,    0,    0

Ahora todos las secuencias tienen la misma longitud (en las más cortas se ha añadido -100 hasta alcanzar la longitud máxima en el lote).

## Modelo


Primero hay que definir las métricas que el modelo debe calcular en cada época sobre el conjunto de validación.

La función compute_metrics() recibirá las predicciones y sus correspondientes etiquetas gold-standard, y calculará las métricas más apropiadas para la tarea (en concreto, precision, recall y f1). La función devuelve un diccionario con los nombres de las métricas y sus puntuaciones.

La librería seqeval permite calcular fácilmente esas métricas para cada uno de las etiquetas (!pip install -q seqeval Evaluate)

In [12]:
import evaluate

metric = evaluate.load("seqeval")

import numpy as np

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Ignoramos los tokens especiales (-100)
    true_labels = [[idx2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [idx2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }


In [13]:
# Se definen el conjunto de hiperparámetros
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir='./outputs/',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=1,  # 1
    weight_decay=0.01,
)

In [14]:
# lo que se está haciendo es un problema de clasificación realmente
from transformers import AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    # instead of using num_labels, for NER is better to provide the correspondence between labels and their indexes
    id2label=idx2label,
    label2id=label2idx,
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
model.config.num_labels


9

## Entrenamiento

Podemos usar la clase Trainer que ayuda mucho a entrenar el modelo. IMPORTANTE: recuerda comprobar que tu entorno de ejecución usa GPU o TPU.

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)
trainer.train()

/tmp/ipykernel_117665/3203677919.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/madrueno/test-transformers/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.081224,0.859099,0.908112,0.882926,0.977601


TrainOutput(global_step=439, training_loss=0.21925343576487757, metrics={'train_runtime': 132.127, 'train_samples_per_second': 106.269, 'train_steps_per_second': 3.323, 'total_flos': 387192806324640.0, 'train_loss': 0.21925343576487757, 'epoch': 1.0})

## Evaluación

In [17]:
# Evaluación sobre el conjunto de validación
trainer.evaluate()

/home/madrueno/test-transformers/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.08122394979000092,
 'eval_precision': 0.8590988696067505,
 'eval_recall': 0.9081117468865701,
 'eval_f1': 0.8829256320052359,
 'eval_accuracy': 0.9776005180432095,
 'eval_runtime': 14.2828,
 'eval_samples_per_second': 227.547,
 'eval_steps_per_second': 7.141,
 'epoch': 1.0}

In [18]:
# Evaluación sobre el conjunto de datos de test

predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [[idx2label[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]

true_labels = [[idx2label[l] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]

results = metric.compute(predictions=true_predictions, references=true_labels)

results


{'LOC': {'precision': 0.8718984420080784,
  'recall': 0.9058752997601919,
  'f1': 0.8885621875918848,
  'number': 1668},
 'MISC': {'precision': 0.5647607934655776,
  'recall': 0.6894586894586895,
  'f1': 0.6209108402822322,
  'number': 702},
 'ORG': {'precision': 0.79490022172949,
  'recall': 0.8633353401565322,
  'f1': 0.8277056277056276,
  'number': 1661},
 'PER': {'precision': 0.9448529411764706,
  'recall': 0.9536178107606679,
  'f1': 0.9492151431209603,
  'number': 1617},
 'overall_precision': 0.8249253235977431,
 'overall_recall': 0.8801345609065155,
 'overall_f1': 0.8516361144423504,
 'overall_accuracy': 0.9636784796961914}

In [19]:
# Resultados a nivel de token

from sklearn.metrics import classification_report
print(classification_report(np.concatenate(true_labels), np.concatenate(true_predictions)))

              precision    recall  f1-score   support

       B-LOC       0.91      0.92      0.91      1668
      B-MISC       0.77      0.76      0.76       702
       B-ORG       0.86      0.89      0.88      1661
       B-PER       0.96      0.96      0.96      1617
       I-LOC       0.87      0.88      0.88      1748
      I-MISC       0.47      0.55      0.51       886
       I-ORG       0.87      0.92      0.90      3172
       I-PER       0.97      0.97      0.97      4082
           O       0.99      0.98      0.99     47925

    accuracy                           0.96     63461
   macro avg       0.85      0.87      0.86     63461
weighted avg       0.97      0.96      0.96     63461



In [20]:
# Resultados a nivel entidad

from seqeval.metrics import classification_report as classification_report_seqeval
print(classification_report_seqeval(true_labels, true_predictions))


              precision    recall  f1-score   support

         LOC       0.87      0.91      0.89      1668
        MISC       0.56      0.69      0.62       702
         ORG       0.79      0.86      0.83      1661
         PER       0.94      0.95      0.95      1617

   micro avg       0.82      0.88      0.85      5648
   macro avg       0.79      0.85      0.82      5648
weighted avg       0.83      0.88      0.85      5648

